# Foundation 14 WALE LES: channel395

## Preamble

In [1]:
# Standard Library
import os
from pathlib import Path
import shutil
import subprocess
from tempfile import mkdtemp

# FoamNordic
import foamnordic as fno

### Directories

In [2]:
# Locate the Foundation tutorial and set the output directory
tutorials_dir = os.environ.get("FOAM_TUTORIALS")
if os.environ.get("WM_PROJECT_VERSION") != "14" or not tutorials_dir:
    raise RuntimeError("Load openfoam-org/14 before starting the notebook kernel")

SOURCE_CASE = Path(tutorials_dir) / "incompressibleFluid/channel395"
if not (SOURCE_CASE / "system/decomposeParDict").is_file():
    raise FileNotFoundError(f"The official channel395 case is missing from {SOURCE_CASE}")

OUTPUT_DIR = Path("output/channel395-of14").resolve()

### Configuration

In [3]:
# Configure a short four-rank Roihu test
ACCOUNT = "<allocation-account>"  # e.g. project_xxxxxxx
PARTITION = "test"
OF_CMD = "module load openfoam-org/14"
N_RANKS = 4
N_STEPS = 5
USE_FOAMNORDIC_WALE = False  # Enable nutFjord with a built OF14 adapter

if ACCOUNT == "<allocation-account>":
    raise ValueError("Set ACCOUNT to your Roihu allocation account")

In [4]:
# Request one CPU per solver rank
scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time="00:05:00",
    openfoam=fno.Slurm.openfoam(nodes=1, ntasks=N_RANKS, cpus_per_task=1, mem_per_cpu="2G"),
    model=None,
)

## Example - WALE LES

### Case Definition

In [5]:
# Shorten the copied case while keeping its time step and decomposition
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
case_dir = Path(mkdtemp(prefix="channel395-", dir=OUTPUT_DIR)) / "channel395"
shutil.copytree(SOURCE_CASE, case_dir)
control_dict = case_dir / "system/controlDict"

delta_t = float(subprocess.check_output(
    ["foamDictionary", str(control_dict), "-entry", "deltaT", "-value"],
    text=True,
))
end_time = N_STEPS * delta_t

for entry, value in {"endTime": end_time, "writeInterval": N_STEPS}.items():
    subprocess.run(
        ["foamDictionary", str(control_dict), "-entry", entry, "-set", str(value)],
        check=True,
        capture_output=True,
        text=True,
    )

In [6]:
# Generate the mesh and use the supplied 1 x 2 x 2 decomposition
case = fno.OpenFOAM.Case(
    case_dir=case_dir,
    run_dir=OUTPUT_DIR,
    of_cmd=OF_CMD,
    shell="bash",
    application="foamRun",
    ranks=N_RANKS,
)
case.initialize(mesh="blockMesh", validate_mesh=True)

### Closure Definition

In [7]:
# Define the WALE eddy viscosity for the optional native FoamNordic closure
def wale_function(velocity_grad, delta, C_w=0.325):
    gradient_squared = fno.Math.matmul(left=velocity_grad, right=velocity_grad)
    traceless_squared = fno.Math.dev(value=fno.Math.symm(value=gradient_squared))
    strain = fno.Math.symm(value=velocity_grad)

    sd2 = fno.Math.ddot(left=traceless_squared, right=traceless_squared)
    s2 = fno.Math.ddot(left=strain, right=strain)
    numerator = sd2**1.5
    denominator = s2**2.5 + sd2**1.25
    positive_denominator = denominator > 0.0
    safe_denominator = fno.Math.where(
        condition=positive_denominator,
        when_true=denominator,
        when_false=1.0,
    )

    nut = (C_w * delta)**2 * numerator / safe_denominator

    return fno.Math.where(condition=positive_denominator, when_true=nut, when_false=0.0)

In [8]:
# Keep the built-in WALE model unless the FoamNordic closure is enabled
closures = ()
if USE_FOAMNORDIC_WALE:
    wale_closure = fno.Closure(
        name="nutFjord",
        operator=fno.Operator.function(function=wale_function),
        inputs={"velocity_grad": fno.Field.grad(value="U"), "delta": fno.Field.delta()},
        outputs={"nut": fno.Field(name="nut")},
    )
    closures = (wale_closure,)

### Submit Job

In [9]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(case=case, closures=closures, scheduler=scheduler, verbose=True)

Resource,OpenFOAM,Model,Allocation
Processes,4 tasks,0 host(s),1 node(s)
CPU,4 cores,0 cores,4 cores
Memory,8 GiB,0 KiB,8 GiB
Field programs,-,0,0
SHM,-,none,none


In [10]:
# Launch the workload
run = longship.launch()

[FoamNordic] Preparing mesh with blockMesh: channel395


[FoamNordic] Mesh is ready: channel395


[FoamNordic] Sailing submitted with Job ID: 3416581


[FoamNordic] Sailing has launched with Job ID: 3416581
[FoamNordic] Sailing started at: 2026-11-25T15:28:10
[FoamNordic] Sailing in background: channel395


In [11]:
# Wait for the job to complete
result = run.stop(timeout=600, progress=True)
if not result.success:
    raise RuntimeError(f"OpenFOAM failed; inspect {result.solver_log}")

In [12]:
# Summarize the job result
result.summary(style="compact", display=True)

Job ID,Name,Status,Partition,Node,Elapsed
3416581,channel395,succeeded,interactive,rc5283,00:00:05


### Post-processing

In [13]:
# Reconstruct the final saved fields from all four processor directories
subprocess.run(
    ["reconstructPar", "-case", str(result.case), "-latestTime"],
    check=True,
    capture_output=True,
    text=True,
);

In [14]:
# Summarize velocity, kinematic pressure and subgrid viscosity
post = result.postprocess
statistics = post.statistics(fields=["U", "p", "nut"], time_idx=-1, verbose=True)

Field,Min,Max,Mean,Std,RMS
U,2.823460e-03,1.893437e-01,1.100895e-01,4.609250e-02,1.193491e-01
p,-4.597530e-04,8.373460e-04,1.643232e-04,9.978009e-05,1.922451e-04
nut,1.614030e-14,1.717690e-04,6.679394e-06,1.159832e-05,1.338415e-05
